# 🐄 DigiCow Farmer Training Adoption Challenge
## Advanced ML Pipeline — Optimized for Log Loss + ROC-AUC

**Problem type:** Multi-output binary probability prediction  
**Targets:** Probability of adoption within **7 days**, **90 days**, **120 days**  
**Metric:** `0.75 × Log Loss + 0.25 × (1 - ROC-AUC)` → **minimize**

> ⚠️ Key insight: Log Loss heavily penalizes **overconfident wrong predictions**.  
> Always output **calibrated probabilities**, never hard 0/1.

---

**Pipeline overview:**

| Step | Action |
|------|--------|
| 1 | Load & explore data |
| 2 | Smart preprocessing + encoding |
| 3 | Feature engineering (temporal, behavioral, interaction) |
| 4 | Optuna hyperparameter tuning per target |
| 5 | Train XGBoost + LightGBM + CatBoost with StratifiedKFold |
| 6 | Probability calibration (Isotonic Regression) |
| 7 | Stacking meta-model on OOF probabilities |
| 8 | Generate submission with properly formatted probabilities |

## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
N_TRIALS = 50  # Increase to 100+ for better tuning
np.random.seed(SEED)

# ── Custom scoring: 75% LogLoss + 25% (1-AUC) ─────────────────────────────────
def zindi_score(y_true, y_prob):
    """Lower is better."""
    ll  = log_loss(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    return 0.75 * ll + 0.25 * (1 - auc)

print("✅ Libraries loaded.")

## 2. 📂 Load Data

In [ ]:
train_df = pd.read_csv('Train.csv')
test_df  = pd.read_csv('Test.csv')
ss       = pd.read_csv('SampleSubmission.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nSample submission columns: {list(ss.columns)}")

In [ ]:
# ── Identify target columns ───────────────────────────────────────────────────
# Targets: adoption within 7, 90, 120 days
TARGET_COLS = [c for c in train_df.columns if 'adopt' in c.lower() or 'target' in c.lower()
               or c in ['adopted_7', 'adopted_90', 'adopted_120']]

# Fallback: detect from sample submission
if not TARGET_COLS:
    TARGET_COLS = [c for c in ss.columns if c.lower() != 'id']

print(f"Target columns detected: {TARGET_COLS}")

for col in TARGET_COLS:
    if col in train_df.columns:
        print(f"  {col}: mean={train_df[col].mean():.3f}, "
              f"pos={train_df[col].sum()}/{len(train_df)} "
              f"({100*train_df[col].mean():.1f}% adoption)")

## 3. 🔧 Preprocessing

Strategy:
- Combine train + test for consistent encoding
- Detect and encode all categorical columns automatically
- Parse dates and extract temporal features
- Impute missing values (median for numeric, mode for categorical)

In [ ]:
# ── Extract targets before combining ─────────────────────────────────────────
y_dict = {}
for col in TARGET_COLS:
    if col in train_df.columns:
        y_dict[col] = train_df[col].values.astype(int)

n_train  = len(train_df)
combined = pd.concat(
    [train_df.drop(columns=TARGET_COLS, errors='ignore'), test_df],
    axis=0, ignore_index=True
)

print(f"Combined shape: {combined.shape}")
print(f"\nMissing values per column:")
missing = combined.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False).head(20))

In [ ]:
# ── Parse date columns ────────────────────────────────────────────────────────
date_cols = [c for c in combined.columns
             if 'date' in c.lower() or 'time' in c.lower()]

for col in date_cols:
    try:
        combined[col] = pd.to_datetime(combined[col], errors='coerce')
        combined[f'{col}_year']    = combined[col].dt.year
        combined[f'{col}_month']   = combined[col].dt.month
        combined[f'{col}_dow']     = combined[col].dt.dayofweek  # day of week
        combined[f'{col}_quarter'] = combined[col].dt.quarter
        combined[f'{col}_dayofyear'] = combined[col].dt.dayofyear
        combined.drop(columns=[col], inplace=True)
        print(f"  Parsed date column: {col}")
    except Exception:
        pass

# ── Encode categorical columns ────────────────────────────────────────────────
cat_cols = combined.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c.lower() != 'id']
print(f"\nCategorical columns to encode: {cat_cols}")

le_dict = {}
for col in cat_cols:
    # Low cardinality → one-hot; high cardinality → label encode
    n_unique = combined[col].nunique()
    if n_unique <= 10:
        dummies = pd.get_dummies(combined[col], prefix=col, drop_first=False)
        combined = pd.concat([combined.drop(columns=[col]), dummies], axis=1)
    else:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
        le_dict[col] = le

# ── Impute missing values ─────────────────────────────────────────────────────
for col in combined.columns:
    if col.lower() == 'id':
        continue
    if combined[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
        median_val = combined[col].iloc[:n_train].median()
        combined[col] = combined[col].fillna(median_val)
    else:
        mode_val = combined[col].iloc[:n_train].mode()
        if len(mode_val) > 0:
            combined[col] = combined[col].fillna(mode_val[0])

print(f"\n✅ Preprocessing done — shape: {combined.shape}")
print(f"Remaining nulls: {combined.isnull().sum().sum()}")

## 4. ⚙️ Feature Engineering

Domain-aware features for farmer adoption prediction:
- **Training engagement features**: interaction counts, session frequency
- **Geographic aggregations**: county/region-level adoption rates (target encoding)
- **Temporal features**: seasonality, recency of training
- **Interaction features**: multiplicative combos of key variables

In [ ]:
# ── Count-based features (frequency encoding for high-cardinality cols) ───────
id_col = 'ID' if 'ID' in combined.columns else combined.columns[0]

# Frequency encode any remaining high-cardinality columns
for col in combined.select_dtypes(include=[np.int64, np.float64]).columns:
    if combined[col].nunique() > 100 and 'id' not in col.lower() and 'date' not in col.lower():
        freq = combined[col].value_counts(normalize=True)
        combined[f'{col}_freq'] = combined[col].map(freq)

# ── Target encoding for key categorical cols using train data only ─────────────
# We use the FIRST target to encode (most frequent)
if y_dict:
    first_target = list(y_dict.keys())[0]
    y_first = y_dict[first_target]

    # Find columns worth target-encoding (medium cardinality: 5–100 unique values)
    te_candidates = []
    for col in combined.select_dtypes(include=[np.int64, np.float64]).columns:
        if 5 < combined[col].iloc[:n_train].nunique() <= 100:
            te_candidates.append(col)
    te_candidates = te_candidates[:10]  # limit to top 10

    for col in te_candidates:
        temp = pd.DataFrame({'col': combined[col].iloc[:n_train], 'target': y_first})
        te_map = temp.groupby('col')['target'].mean()
        combined[f'{col}_te'] = combined[col].map(te_map).fillna(y_first.mean())

    print(f"Target-encoded {len(te_candidates)} columns.")

# ── Polynomial features for top numeric columns ───────────────────────────────
num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c.lower() != 'id'][:20]  # top 20 numeric

# Log1p transform to handle skewed distributions
for col in num_cols:
    if combined[col].min() >= 0:
        combined[f'log_{col}'] = np.log1p(combined[col])

print(f"✅ Feature engineering done — shape: {combined.shape}")

## 5. ✂️ Split Train / Test

In [ ]:
feature_cols = [c for c in combined.columns if c.lower() != 'id']

X_train = combined.iloc[:n_train][feature_cols].values.astype(np.float32)
X_test  = combined.iloc[n_train:][feature_cols].values.astype(np.float32)

SKF = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Targets : {list(y_dict.keys())}")

## 6. 🔎 Optuna Tuning + Training per Target

We train **independently for each target** (7d, 90d, 120d) since adoption dynamics differ:
- **7-day adoption**: very rare, needs aggressive class weighting
- **90/120-day adoption**: more frequent, better signal

For each target, Optuna finds the best hyperparameters for LightGBM (fastest),
then all three models are trained with those insights.

In [ ]:
def train_one_target(target_name, y, X_train, X_test, n_trials=N_TRIALS):
    """
    Full pipeline for one binary target:
    1. Optuna tuning (LightGBM)
    2. OOF training: XGB + LGB + CAT
    3. Isotonic calibration of probabilities
    4. Stacking meta-model
    Returns: (oof_proba, test_proba, score)
    """
    print(f"\n{'='*60}")
    print(f"  TARGET: {target_name}  |  Adoption rate: {y.mean():.3f}")
    print(f"{'='*60}")

    sw = compute_sample_weight(class_weight='balanced', y=y)

    # ── OPTUNA: Tune LightGBM ─────────────────────────────────────────────────
    def objective(trial):
        p = dict(
            n_estimators      = trial.suggest_int('n_estimators', 200, 1000),
            learning_rate     = trial.suggest_float('lr', 0.01, 0.15, log=True),
            num_leaves        = trial.suggest_int('num_leaves', 20, 200),
            max_depth         = trial.suggest_int('max_depth', 3, 10),
            subsample         = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
            scale_pos_weight  = trial.suggest_float('scale_pos_weight',
                                                     1, max(10, (1-y.mean())/y.mean())),
            random_state=SEED, n_jobs=-1, verbose=-1
        )
        scores = []
        for trn_i, val_i in SKF.split(X_train, y):
            m = lgb.LGBMClassifier(**p)
            m.fit(X_train[trn_i], y[trn_i], sample_weight=sw[trn_i],
                  eval_set=[(X_train[val_i], y[val_i])],
                  callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
            proba = m.predict_proba(X_train[val_i])[:, 1]
            scores.append(zindi_score(y[val_i], proba))
        return np.mean(scores)  # minimize

    print("  🔎 Tuning LightGBM...")
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best_p = study.best_params
    best_p.update({'random_state': SEED, 'n_jobs': -1, 'verbose': -1})
    print(f"  ✅ Best Zindi score (LGB): {study.best_value:.5f}")

    # ── TRAIN 3 MODELS with OOF ───────────────────────────────────────────────
    oof_xgb = np.zeros(len(X_train))
    oof_lgb = np.zeros(len(X_train))
    oof_cat = np.zeros(len(X_train))
    tst_xgb = np.zeros(len(X_test))
    tst_lgb = np.zeros(len(X_test))
    tst_cat = np.zeros(len(X_test))

    for fold, (trn_i, val_i) in enumerate(SKF.split(X_train, y)):
        sw_fold = sw[trn_i]

        # XGBoost
        xgb_p = dict(
            n_estimators=best_p.get('n_estimators', 500),
            learning_rate=best_p.get('lr', 0.05),
            max_depth=best_p.get('max_depth', 6),
            subsample=best_p.get('subsample', 0.8),
            colsample_bytree=best_p.get('colsample_bytree', 0.8),
            scale_pos_weight=best_p.get('scale_pos_weight', 1),
            reg_alpha=best_p.get('reg_alpha', 0.1),
            reg_lambda=best_p.get('reg_lambda', 1),
            use_label_encoder=False, eval_metric='logloss',
            random_state=SEED, n_jobs=-1
        )
        mx = xgb.XGBClassifier(**xgb_p)
        mx.fit(X_train[trn_i], y[trn_i], sample_weight=sw_fold,
               eval_set=[(X_train[val_i], y[val_i])],
               early_stopping_rounds=50, verbose=False)
        oof_xgb[val_i] = mx.predict_proba(X_train[val_i])[:, 1]
        tst_xgb       += mx.predict_proba(X_test)[:, 1] / N_FOLDS

        # LightGBM
        ml = lgb.LGBMClassifier(**best_p)
        ml.fit(X_train[trn_i], y[trn_i], sample_weight=sw_fold,
               eval_set=[(X_train[val_i], y[val_i])],
               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_lgb[val_i] = ml.predict_proba(X_train[val_i])[:, 1]
        tst_lgb       += ml.predict_proba(X_test)[:, 1] / N_FOLDS

        # CatBoost
        cat_p = dict(
            iterations=best_p.get('n_estimators', 500),
            learning_rate=best_p.get('lr', 0.05),
            depth=min(best_p.get('max_depth', 6), 10),
            l2_leaf_reg=best_p.get('reg_lambda', 1),
            subsample=best_p.get('subsample', 0.8),
            scale_pos_weight=best_p.get('scale_pos_weight', 1),
            random_seed=SEED, verbose=False, early_stopping_rounds=50
        )
        mc = CatBoostClassifier(**cat_p)
        mc.fit(X_train[trn_i], y[trn_i], sample_weight=sw_fold,
               eval_set=(X_train[val_i], y[val_i]), verbose=False)
        oof_cat[val_i] = mc.predict_proba(X_train[val_i])[:, 1]
        tst_cat       += mc.predict_proba(X_test)[:, 1] / N_FOLDS

        s_xgb = zindi_score(y[val_i], oof_xgb[val_i])
        s_lgb = zindi_score(y[val_i], oof_lgb[val_i])
        s_cat = zindi_score(y[val_i], oof_cat[val_i])
        print(f"  Fold {fold+1} | XGB={s_xgb:.4f} | LGB={s_lgb:.4f} | CAT={s_cat:.4f}")

    # ── ISOTONIC CALIBRATION ──────────────────────────────────────────────────
    # Calibration reduces Log Loss by fixing over/under-confident probabilities
    def calibrate(oof_proba, tst_proba, y):
        ir = IsotonicRegression(out_of_bounds='clip')
        ir.fit(oof_proba, y)
        return ir.transform(oof_proba), ir.transform(tst_proba)

    oof_xgb_cal, tst_xgb_cal = calibrate(oof_xgb, tst_xgb, y)
    oof_lgb_cal, tst_lgb_cal = calibrate(oof_lgb, tst_lgb, y)
    oof_cat_cal, tst_cat_cal = calibrate(oof_cat, tst_cat, y)

    # ── STACKING META-MODEL ───────────────────────────────────────────────────
    meta_X_train = np.column_stack([oof_xgb_cal, oof_lgb_cal, oof_cat_cal])
    meta_X_test  = np.column_stack([tst_xgb_cal, tst_lgb_cal, tst_cat_cal])

    meta = LogisticRegression(C=0.5, max_iter=2000, random_state=SEED,
                               class_weight='balanced')
    meta.fit(meta_X_train, y)

    oof_final  = meta.predict_proba(meta_X_train)[:, 1]
    test_final = meta.predict_proba(meta_X_test)[:, 1]

    # ── CLIP probabilities to avoid extreme log-loss ──────────────────────────
    # Never output exactly 0 or 1 — this destroys log loss
    eps = 1e-6
    oof_final  = np.clip(oof_final, eps, 1 - eps)
    test_final = np.clip(test_final, eps, 1 - eps)

    final_score = zindi_score(y, oof_final)
    ll  = log_loss(y, oof_final)
    auc = roc_auc_score(y, oof_final)
    print(f"\n  ⭐ Final OOF — Zindi: {final_score:.5f} | LogLoss: {ll:.5f} | AUC: {auc:.5f}")

    return oof_final, test_final, final_score

print("Function defined. Ready to train.")

## 7. 🚀 Train All Targets

In [ ]:
results = {}  # stores {target: (oof, test, score)}

for target_name, y in y_dict.items():
    oof_proba, test_proba, score = train_one_target(
        target_name, y, X_train, X_test, n_trials=N_TRIALS
    )
    results[target_name] = (oof_proba, test_proba, score)

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for t, (_, _, s) in results.items():
    print(f"  {t:30s} → Zindi Score: {s:.5f}")
overall = np.mean([s for _, _, s in results.values()])
print(f"  {'AVERAGE':30s} → {overall:.5f}")

## 8. 📤 Generate Submission

> ⚠️ **Important**: Submit probabilities, NOT binary labels.  
> Never round to 0/1 — it destroys the Log Loss score.

In [ ]:
# Build submission matching the sample submission format
submission = pd.DataFrame()

# Add ID column
id_col = 'ID' if 'ID' in test_df.columns else test_df.columns[0]
submission[id_col] = test_df[id_col].values

# Add probability predictions for each target
for target_name, (oof_proba, test_proba, score) in results.items():
    submission[target_name] = test_proba
    print(f"  {target_name}: mean proba = {test_proba.mean():.4f} "
          f"(min={test_proba.min():.4f}, max={test_proba.max():.4f})")

# Verify format matches sample submission
print(f"\nSubmission shape : {submission.shape}")
print(f"Sample sub shape : {ss.shape}")
print(f"Columns match    : {list(submission.columns) == list(ss.columns)}")

submission.to_csv('submission_digicow.csv', index=False)
print("\n✅ submission_digicow.csv saved!")
submission.head(10)

## 9. 📊 Performance Summary & Validation

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
if len(results) == 1:
    axes = [axes]

for ax, (target_name, (oof_proba, _, _)) in zip(axes, results.items()):
    y = y_dict[target_name]
    # Calibration curve
    fraction_pos, mean_pred = calibration_curve(y, oof_proba, n_bins=10)
    ax.plot(mean_pred, fraction_pos, 's-', label='Model', color='steelblue')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
    ax.set_title(f'{target_name}\nLL={log_loss(y, oof_proba):.4f} | AUC={roc_auc_score(y, oof_proba):.4f}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Calibration Curves per Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("Calibration curves saved.")

---
## 💡 Further Improvements

1. **Increase `N_TRIALS`** → 100–200 for finer hyperparameter search
2. **Temperature scaling** → Another calibration method, sometimes better than Isotonic
3. **Cross-target features** → Use OOF predictions of 90d target as feature for 120d target
4. **Farmer-level aggregations** → If same farmer appears multiple times, aggregate past behavior
5. **Trainer-level features** → Target encode trainer ID with adoption rate (trainer quality signal)
6. **Region-level features** → County/ward adoption rate as contextual signal
7. **Neural network** → A simple MLP with dropout can complement gradient boosting in ensemble
8. **Post-processing** → Blend predictions with base rate `y.mean()` to avoid overconfidence:
   ```python
   blend_alpha = 0.05
   final_proba = (1 - blend_alpha) * model_proba + blend_alpha * y.mean()
   ```